## Imports

In [ ]:
# =============================================================
# Gemeinsame Bibliotheken für alle vier CRISP-DM-Phasen 
# (Data Understanding, Data Preparation, Modeling, 
# Demonstration und Evaluation).
# =============================================================

# Standardbibliothek
import os
import json

# Datenverarbeitung
import numpy as np
import pandas as pd

# Visualisierung
import matplotlib.pyplot as plt
import seaborn as sns

# Process Mining
import pm4py

# Persistierung von Modellen und Encodern
import joblib

# Machine Learning
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix
)

# Modellinterpretation
import shap

## Data Understanding

In [ ]:
"""
Umsetzung der in Kapitel 4.1.1 beschriebenen Methodik:
- Datenimport aus Excel
- Univariate und bivariate Analysen (EDA)
- Datenqualitätsprüfung: Fehlwerte und Ausreißer in Gesamtdurchlaufzeit und Wartezeiten
- Prozessvarianten

Output: Visualisierungen sowie ein MT-gefilterter Datensatz als Übergabe 
an die Data-Preparation-Phase
"""


# =============================================================
# Konfiguration
# 
# Pfadangaben relativ zum Notebook-Speicherort. Die Daten- und 
# Output-Verzeichnisse müssen vor der Ausführung angelegt werden.
# =============================================================
FILE_PATH = "<pfad-zur-eingangsdatei>/finale_daten.xlsx"
OUTPUT_DIR = "<pfad-zum-output-verzeichnis>"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Meilensteinkette
MEILENSTEINE = ["SOF", "PF", "SOP", "EOP", "RFS"]
PLAN_SPALTEN = [f"{m} PLAN" for m in MEILENSTEINE]
IST_SPALTEN = [f"{m} ACT" for m in MEILENSTEINE]
DELTA_SPALTEN = [f"{m} Δ" for m in MEILENSTEINE]

# Vorberechnete Wartezeiten zwischen aufeinanderfolgenden Meilensteinen (Ist in der Excel)
WARTEZEIT_SPALTEN = [
    "Wartezeit SOF ACT - Auftrag",
    "Wartezeit PF ACT - SOF ACT",
    "Wartezeit SOP ACT - PF ACT",
    "Wartezeit EOP ACT - SOP ACT",
    "Wartezeit RFS ACT - EOP ACT",
]

# Vorberechnete Gesamtdurchlaufzeit bis RFS ACT (Ist in der Excel: RFS ACT - Auftragsanlagedatum)
DURCHLAUFZEIT_GESAMT = "Durchlaufzeit bis RFS ACT"


# =============================================================
# 1. Datenimport und initiale Filterung
# 
# Reduktion des Rohdatensatzes auf den Untersuchungsgegenstand der 
# Arbeit: abgeschlossene BTO-Aufträge der Business Division MT.
# =============================================================

df = pd.read_excel(FILE_PATH)
print(f"Shape (Rohdaten): {df.shape}")

# Identifikation noch nicht abgeschlossener Aufträge über das Fehlen 
# eines RFS Δ-Werts. Diese sind nicht Untersuchungsgegenstand, da die 
# Zielvariable für sie nicht definiert ist.
offene_auftraege = df["RFS Δ"].isna().sum()
print(f"Info: {offene_auftraege} Aufträge ohne RFS Δ werden gefiltert.")
df = df.dropna(subset=["RFS Δ"]).copy()
print(f"Shape (nach RFS Δ-Filter): {df.shape}")

# Filterung auf Business Division MT (Fokus der Arbeit)
df = df[df["Business Division"] == "MT"].reset_index(drop=True)
print(f"Shape (nach MT-Filter): {df.shape}")

# Fertigung Freigabe wird von pandas als object identifiziert, obwohl es in Excel als Datum formatiert ist
df["Fertigung Freigabe"] = pd.to_datetime(df["Fertigung Freigabe"], errors="coerce")
 
print(df.dtypes)
df.head()


# =============================================================
# 2. Univariate Analyse der Zielvariable RFS Δ
# 
# Quantifizierung der Verteilung als Grundlage für die Klassendefinition 
# und das Verständnis der Verspätungsmuster im Datensatz.
# =============================================================

print(df["RFS Δ"].describe())

fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(df["RFS Δ"].dropna(), bins=50, kde=True, ax=ax)
# Vertikale Linie bei Δ = 0 markiert den betrieblichen Zielzustand 
# „pünktlich" als Referenzpunkt für die Verteilungsinterpretation.
ax.axvline(0, color="red", linestyle="--", label="pünktlich (Δ=0)")
ax.set_xlabel("RFS Δ [Kalendertage]")
ax.set_ylabel("Häufigkeit")
ax.set_title("Verteilung der Zielvariable RFS Δ")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "01_verteilung_RFS.png"), dpi=150)
plt.show()

# Klassenverteilung (3 Klassen gemäß Hauptklassifikation)
klassen = df["Klassen"].value_counts(dropna=False)
print(klassen)
print((klassen / klassen.sum() * 100).round(2))

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df.dropna(subset=["Klassen"]), x="Klassen",
              order=["früh", "pünktlich", "verspätet"],
              color="steelblue", ax=ax)
ax.set_ylabel("Anzahl Aufträge")
ax.set_title("Klassenverteilung")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "02_klassenverteilung.png"), dpi=150)
plt.show()


# =============================================================
# 3. Bivariate Analyse: Zusammenhang vorgelagerter Δ-Werte mit RFS Δ
# 
# Berechnung von Pearson- und Spearman-Korrelationen, um zu prüfen, 
# inwiefern Abweichungen vorgelagerter Meilensteine mit der finalen 
# Zielvariable korrelieren. Spearman wird ergänzend zu Pearson 
# verwendet, da monotone, aber nicht zwingend lineare Zusammenhänge 
# in Prozessdaten auftreten können.
# =============================================================

print("Korrelationen mit RFS Δ:")
for sp in ["SOF Δ", "PF Δ", "SOP Δ", "EOP Δ"]:
    # Listenweiser Fallausschluss pro Spaltenpaar: nur Aufträge mit 
    # Werten in beiden Spalten gehen in die Berechnung ein. Die 
    # Stichprobengröße variiert daher je Paar.
    paar = df[[sp, "RFS Δ"]].dropna()
    n = len(paar)
    pear = paar[sp].corr(paar["RFS Δ"], method="pearson")
    spear = paar[sp].corr(paar["RFS Δ"], method="spearman")
    print(f"  {sp}: n={n:>5}  Pearson={pear:+.3f}  Spearman={spear:+.3f}")

# Spearman-Korrelationsmatrix der Δ-Spalten als Heatmap.
corr_spearman = df[DELTA_SPALTEN].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_spearman, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Spearman-Korrelation der Δ-Spalten")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_korrelation_heatmap_SPEARMAN.png"), dpi=150)
plt.show()

# Pearson-Korrelationsmatrix als Vergleichsmaß.
corr_pearson = df[DELTA_SPALTEN].corr(method="pearson")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_pearson, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson-Korrelation der Δ-Spalten")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_korrelation_heatmap_PEARSON.png"), dpi=150)
plt.show()

# Berechnung der paarweisen Stichprobengrößen für die Korrelations-
# matrix. Da Korrelationen pro Spaltenpaar nur auf gemeinsam vorhan-
# denen Beobachtungen berechnet werden, dokumentiert diese Matrix die 
# unterschiedlichen Stichprobengrößen je Korrelationswert.
maske = df[DELTA_SPALTEN].notna().astype(int)
n_matrix = maske.T.dot(maske)
print("n-Matrix (paarweise gültige Beobachtungen):")
print(n_matrix)


# =============================================================
# 4a. Datenqualität: Prüfung auf Fehlwerte
# 
# Identifikation systematischer Erfassungslücken pro Spalte.
# =============================================================

fehlend = df.isnull().sum()
print(pd.DataFrame({
    "Anzahl": fehlend,
    "Prozent": (fehlend / len(df) * 100).round(2)
}))

if fehlend.sum() > 0:
    fig, ax = plt.subplots(figsize=(8, 10))
    fehlend[fehlend > 0].sort_values().plot(kind="barh", ax=ax, color="tomato")
    ax.set_xlabel("Anzahl fehlender Werte")
    ax.set_title("Fehlende Werte pro Spalte")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "05_fehlwerte.png"), dpi=150)
    plt.show()


# =============================================================
# 4b. Datenqualität: Ausreißererkennung mittels Tukey-Methode
# 
# Zweistufige Anwendung der Tukey-Methode (Faktor 1,5) gemäß der in 
# Kapitel 4.1.1 beschriebenen Methodik:
# - Stufe 1 (Makroebene): Gesamtdurchlaufzeit bis RFS ACT
# - Stufe 2 (Mikroebene): einzelne Wartezeiten zwischen den Meilensteinen
# =============================================================

# Stufe 1 (Makroebene): Gesamtdurchlaufzeit bis RFS ACT.
# Ziel ist die Identifikation auffälliger Durchlaufzeiten.
print("Stufe 1: Ausreißer in der Gesamtdurchlaufzeit (bis RFS ACT):\n")
q1, q3 = df[DURCHLAUFZEIT_GESAMT].quantile([0.25, 0.75])
iqr = q3 - q1
unten, oben = q1 - 1.5 * iqr, q3 + 1.5 * iqr
maske_makro = (df[DURCHLAUFZEIT_GESAMT] < unten) | (df[DURCHLAUFZEIT_GESAMT] > oben)
print(f"  {DURCHLAUFZEIT_GESAMT}: Grenzen [{unten:.1f}, {oben:.1f}]  "
      f"Ausreisser: {maske_makro.sum()}")
 
fig, ax = plt.subplots(figsize=(6, 5))
df[[DURCHLAUFZEIT_GESAMT]].boxplot(ax=ax)
ax.set_ylabel("Durchlaufzeit [Kalendertage]")
ax.set_title("Boxplot der Gesamtdurchlaufzeit bis RFS ACT")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "06_boxplot_gesamtdurchlaufzeit.png"), dpi=150)
plt.show()
 
# Stufe 2 (Mikroebene): Wartezeiten zwischen aufeinanderfolgenden Meilensteinen.
# Dieser Schritt erfasst lokale Ausreißer.
print("\nStufe 2: Ausreißer in den Wartezeiten:\n")
for sp in WARTEZEIT_SPALTEN:
    q1, q3 = df[sp].quantile([0.25, 0.75])
    iqr = q3 - q1
    unten, oben = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((df[sp] < unten) | (df[sp] > oben)).sum()
    print(f"  {sp}: Grenzen [{unten:.1f}, {oben:.1f}]  Ausreißer: {n}")
 
fig, ax = plt.subplots(figsize=(10, 5))
df[WARTEZEIT_SPALTEN].boxplot(ax=ax)
ax.set_ylabel("Wartezeit [Kalendertage]")
ax.set_title("Boxplots der Wartezeiten zwischen aufeinanderfolgenden Meilensteinen")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "07_boxplots_wartezeiten.png"), dpi=150)
plt.show()
 


# =============================================================
# 5. Bedingte Wahrscheinlichkeiten P(RFS-Klasse | vorgelagerte Δ-Klasse)
# 
# Ergänzung der Korrelationsanalyse durch die Untersuchung der 
# Klassenverteilung der Zielvariable in Abhängigkeit vorgelagerter 
# Meilensteinklassen. Ziel ist die Beurteilung, inwiefern eine 
# Abweichung an einem vorgelagerten Meilenstein eine entsprechende 
# Abweichung am Zielmeilenstein RFS impliziert.
# 
# Methodische Anmerkung: Die Klassifizierung erfolgt mit derselben 
# Tagesschwelle wie für die Zielvariable RFS Δ, um eine konsistente 
# Vergleichbarkeit zwischen vorgelagerten Klassen und Zielklasse 
# herzustellen.
# =============================================================

def klassifiziere(x):
    if pd.isna(x):
        return np.nan
    if x < 0:
        return "früh"
    if x == 0:
        return "pünktlich"
    return "verspätet"

for sp in ["SOF Δ", "PF Δ", "SOP Δ", "EOP Δ"]:
    vorgelagert = df[sp].apply(klassifiziere)
    # Absolute und prozentuale Cross-Tabulation. Die Normalisierung 
    # erfolgt zeilenweise (normalize="index"), sodass die Werte 
    # jeder Zeile zu 100 % aufsummieren und damit die bedingte 
    # Wahrscheinlichkeit P(RFS-Klasse | vorgelagerte Klasse) 
    # darstellen.
    ctab_abs = pd.crosstab(vorgelagert, df["Klassen"])
    ctab_pct = pd.crosstab(vorgelagert, df["Klassen"], normalize="index") * 100
    n_total = ctab_abs.values.sum()
    n_missing = df[sp].isna().sum()

    print(f"\n=== {sp} ===")
    print(f"n verwendet = {n_total}   |   n fehlend = {n_missing}")
    print("\nAbsolute Häufigkeiten:")
    print(ctab_abs)
    print(f"\nP(Klassen | {sp}-Klasse) in %:")
    print(ctab_pct.round(1))


# =============================================================
# 6. Prozessvarianten mit pm4py
# Konstruktion eines Event-Logs aus dem tabellarischen Datensatz und 
# Identifikation der Trace-Varianten. Ziel ist die Bestimmung des 
# Standardpfads sowie die Aufdeckung chronologischer Inkonsistenzen.
#
# Case-ID = Equipment (Maschine ist einmalig)
# Kettenanfang ist das Auftragsanlagedatum (ohne Plandatum), danach die
# Haupt-Meilensteinkette
# =============================================================

events = []
for _, row in df.iterrows():
    eq = row["Equipment"]
    if pd.isna(eq):
        continue
    # Iteration über die geplante Meilensteinkette: Auftragsanlage 
    # als Beginn, gefolgt von den Ist-Meilensteinen.
    for sp in ["Auftragsanlagedatum"] + IST_SPALTEN:
        if pd.notna(row[sp]):
            # Aktivitätsname: "Auftragsanlage" für das Auftragsanlage-
            # datum, sonst der Meilensteinname ohne den ACT-Suffix.
            aktivitaet = ("Auftragsanlage" if sp == "Auftragsanlagedatum"
                          else sp.replace(" ACT", ""))
            events.append({
                "case:concept:name": str(eq),
                "concept:name": aktivitaet,
                "time:timestamp": pd.to_datetime(row[sp])
            })

# Sortierung nach Case-ID und Zeitstempel ist erforderlich, damit 
# pm4py die Aktivitätenreihenfolge je Case korrekt rekonstruiert.
eventlog = pd.DataFrame(events).sort_values(["case:concept:name", "time:timestamp"])
print(f"Event-Log: {len(eventlog)} Events, "
      f"{eventlog['case:concept:name'].nunique()} Cases")

# Überführung in das pm4py-Standardformat
log = pm4py.format_dataframe(eventlog,
                              case_id="case:concept:name",
                              activity_key="concept:name",
                              timestamp_key="time:timestamp")

varianten = pm4py.get_variants(log)
print(f"Anzahl unterschiedlicher Trace-Varianten: {len(varianten)}")

# Sortierung der Varianten nach absteigender Häufigkeit für die 
# Visualisierung der Top 10.
haeufigkeit = list(varianten.items())
haeufigkeit.sort(key=lambda x: x[1], reverse=True)

print("\nTop 10 Prozessvarianten:\n")
for i, (pfad, n) in enumerate(haeufigkeit[:10], 1):
    pfad_str = " → ".join(pfad) if isinstance(pfad, (list, tuple)) else str(pfad)
    print(f"  {i:2d}. ({n:>5}x)  {pfad_str}")

fig, ax = plt.subplots(figsize=(9, 5))
top_n = min(10, len(haeufigkeit))
ax.bar([f"V{i+1}" for i in range(top_n)],
       [haeufigkeit[i][1] for i in range(top_n)],
       color="steelblue")
ax.set_xlabel("Variante")
ax.set_ylabel("Anzahl Fälle")
ax.set_title("Top-10 Prozessvarianten nach Häufigkeit")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_prozessvarianten.png"), dpi=150)
plt.show()


# =============================================================
# 7. Export des MT-gefilterten Datensatzes
# 
# Übergabepunkt zwischen Data-Understanding- und Data-Preparation-Phase. 
# Der Datensatz ist auf die Business Division MT eingegrenzt und auf 
# abgeschlossene Aufträge reduziert, jedoch inhaltlich unverändert. 
# Bereinigungs- und Transformationsschritte erfolgen in der nach-
# folgenden Phase (Kapitel 5.3).
# =============================================================

EXPORT_PFAD = os.path.join(OUTPUT_DIR, "01_data_understanding_output.xlsx")
df.to_excel(EXPORT_PFAD, index=False)
print(f"Datensatz exportiert: {EXPORT_PFAD}")
print(f"Shape: {df.shape}")

## Data Preparation 

In [ ]:
"""
Umsetzung der in Kapitel 4.1.2 beschriebenen Methodik:
- Listenweiser Fallausschluss bei Fehlwerten
- Filterung auf die Standard-Prozessvariante (Auftragsanlage -> SOF -> 
  PF -> SOP -> EOP -> RFS) als Konsequenz aus der Variantenanalyse 
  in Kapitel 5.2
- Entfernung negativer Wartezeiten als Systemfehler
- Beibehaltung statistischer Ausreißer nach oben als reale 
  Prozessabweichungen
- Aggregation seltener Ausprägungen kategorialer Features zu "other"
- One-Hot-Encoding der kategorialen Features
- Fachlich fundierte Feature-Selektion

Input:  01_data_understanding_output.xlsx (aus Kapitel 5.2)
Output: 02_data_preparation_output.xlsx als Übergabe an die 
        Modeling-Phase (Kapitel 5.4)
"""


# =============================================================
# Konfiguration
# =============================================================

INPUT_PFAD = "<pfad-zum-output-verzeichnis>/01_data_understanding_output.xlsx"
OUTPUT_DIR = "<pfad-zum-output-verzeichnis>"

# Ist-Meilensteine in chronologischer Reihenfolge
IST_SPALTEN = ["SOF ACT", "PF ACT", "SOP ACT", "EOP ACT", "RFS ACT"]

# Wartezeiten zwischen aufeinanderfolgenden Ist-Meilensteinen
# (negative Werte = Systemfehler, werden entfernt)
WARTEZEIT_SPALTEN = [
    "Wartezeit SOF ACT - Auftrag",
    "Wartezeit PF ACT - SOF ACT",
    "Wartezeit SOP ACT - PF ACT",
    "Wartezeit EOP ACT - SOP ACT",
    "Wartezeit RFS ACT - EOP ACT",
]

# Kategoriale Features für One-Hot-Encoding (fachliche Auswahl)
KATEGORIALE_FEATURES = [
    "Plant",
    "Productfamily",
    "Materialnummer",
    "Sales Organization",
]

# Schwellwert für seltene Ausprägungen kategorialer Attribute.
# Ausprägungen, die weniger als 10-mal im Datensatz vorkommen, werden
# zum gemeinsamen Wert "other" aggregiert (Teinemaa et al. 2018).
MIN_HAEUFIGKEIT = 10

# Numerische Features für den finalen Merkmalsvektor. Die Wartezeit 
# RFS ACT - EOP ACT sowie die Gesamtdurchlaufzeit werden bewusst 
# ausgeschlossen, da beide den Zeitpunkt RFS ACT enthalten und damit 
# Zukunftsinformation gegenüber jedem Vorhersagezeitpunkt vor RFS 
# darstellen würden (Data Leakage)
NUMERISCHE_FEATURES = [
    "SOF Δ", "PF Δ", "SOP Δ", "EOP Δ",
    "Wartezeit SOF ACT - Auftrag",
    "Wartezeit PF ACT - SOF ACT",
    "Wartezeit SOP ACT - PF ACT",
    "Wartezeit EOP ACT - SOP ACT",
    "Wartezeit Freigabe - Erfassung",
    "Wartezeit SOP ACT - Freigabe",
    "Delta SOF ACT - Anzahlung",
]

# Zielvariable
ZIELVARIABLE = "Klassen"


# =============================================================
# 1. Datenimport
# =============================================================

df = pd.read_excel(INPUT_PFAD)
print(f"Shape Eingangsdatensatz: {df.shape}")


# =============================================================
# 2. Filterung auf den Standard-Prozesspfad
# 
# Konsequenz aus der Variantenanalyse in Kapitel 5.2: Nur Aufträge, die 
# dem vollständigen Standardpfad Auftragsanlage -> SOF -> PF -> SOP -> 
# EOP -> RFS folgen, werden für die Modellierung verwendet. Damit wird 
# die Vergleichbarkeit der Prozessverläufe sichergestellt und 
# Verzerrungen durch unvollständig erfasste frühe Meilensteine 
# vermieden.
# =============================================================

# Bedingung 1: Vollständigkeit der Zeitstempel entlang der Kette
zeitspalten = ["Auftragsanlagedatum"] + IST_SPALTEN
maske_vollstaendig = df[zeitspalten].notna().all(axis=1)

# Bedingung 2: Prozesslogisch korrekte Reihenfolge der Ist-Zeitstempel. 
# Aufträge, deren Buchungen die erwartete Chronologie verletzen 
# (z.B. RFS vor EOP), werden ebenfalls ausgeschlossen.
maske_reihenfolge = (
    (df["Auftragsanlagedatum"] <= df["SOF ACT"])
    & (df["SOF ACT"] <= df["PF ACT"])
    & (df["PF ACT"] <= df["SOP ACT"])
    & (df["SOP ACT"] <= df["EOP ACT"])
    & (df["EOP ACT"] <= df["RFS ACT"])
)

df = df[maske_vollstaendig & maske_reihenfolge].reset_index(drop=True)
print(f"Shape nach Filter Standard-Variante: {df.shape}")


# =============================================================
# 3. Entfernung negativer Wartezeiten als Systemfehler
# 
# Negative Wartezeiten zwischen aufeinanderfolgenden Prozessschritten 
# sind prozesslogisch ausgeschlossen und werden als Buchungsfehler 
# behandelt. Statistische Ausreißer nach oben verbleiben dagegen im 
# Datensatz, da sie reale (wenn auch verlängerte) Prozessverläufe 
# abbilden können.
# 
# Methodische Ausnahme: Das Attribut "Delta SOF ACT - Anzahlung" wird 
# nicht auf negative Werte gefiltert. In einzelnen Vertriebsregionen 
# erfolgt der Anzahlungseingang nach SOF, sodass negative Werte dort 
# fachlich legitim sind. In anderen Vertriebsregionen muss der Anzahlungseingang 
# vor SOF passieren.
# =============================================================

wartezeiten_pruefen = WARTEZEIT_SPALTEN + [
    "Wartezeit Freigabe - Erfassung",
    "Wartezeit SOP ACT - Freigabe",
]
maske_keine_neg_wartezeiten = (df[wartezeiten_pruefen] >= 0).all(axis=1)
print(f"Aufträge mit negativen Wartezeiten: "
      f"{(~maske_keine_neg_wartezeiten).sum()}")
df = df[maske_keine_neg_wartezeiten].reset_index(drop=True)
print(f"Shape nach Entfernung negativer Wartezeiten: {df.shape}")


# =============================================================
# 4. Listenweiser Fallausschluss verbleibender Fehlwerte
# 
# Auch nach der Filterung auf den Standardpfad können in nicht-zeit-
# bezogenen Spalten (z.B. Sales Organization) noch Fehlwerte vorliegen. 
# Diese werden ebenfalls ausgeschlossen, um einen vollständigen 
# Datensatz für das spätere Modelltraining zu gewährleisten.
# =============================================================

relevante_spalten = NUMERISCHE_FEATURES + KATEGORIALE_FEATURES + [ZIELVARIABLE]

print("\n--- Fehlwerte in relevanten Spalten ---")
fehlend = df[relevante_spalten].isnull().sum()
fehlend = fehlend[fehlend > 0]
if len(fehlend) > 0:
    print(fehlend)
    n_vorher = len(df)
    df = df.dropna(subset=relevante_spalten).reset_index(drop=True)
    print(f"Entfernte Aufträge mit Fehlwerten: {n_vorher - len(df)}")
    print(f"Shape nach Fehlwert-Bereinigung: {df.shape}")
else:
    print("Keine Fehlwerte in relevanten Spalten.")


## =============================================================
# 5. Wiederholung der explorativen Datenanalyse auf dem 
#    bereinigten Datensatz
# 
# Charakterisierung des bereinigten Datensatzes vor dem Feature 
# Engineering. In Kapitel 5.2 ist die Korrelations- und 
# Wahrscheinlichkeitsbetrachtung bereits auf dem ungefilterten 
# Datensatz durchgeführt worden. Die Wiederholung an dieser Stelle erlaubt 
# einen Vergleich, der zeigt, wie sich die Befunde nach dem 
# Filtern auf den Standardpfad verändern.
# =============================================================

print("\n--- Klassenverteilung nach Bereinigung ---")
klassen = df[ZIELVARIABLE].value_counts()
print(klassen)
print((klassen / klassen.sum() * 100).round(2))

fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x=ZIELVARIABLE,
              order=["früh", "pünktlich", "verspätet"],
              color="steelblue", ax=ax)
ax.set_ylabel("Anzahl Aufträge")
ax.set_title("Klassenverteilung nach Bereinigung")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "08_klassenverteilung_bereinigt.png"), dpi=150)
plt.show()

# Deskriptive Statistik der elf numerischen Features
print("\n--- Deskriptive Statistik der numerischen Features ---")
print(df[NUMERISCHE_FEATURES].describe().round(2))

# Korrelationen zwischen den vorgelagerten Δ-Werten und der 
# Zielvariable RFS Δ als Vergleich zum ungefilterten Datensatz 
# (vgl. Kapitel 5.2).
print("Korrelationen mit RFS Δ:")
for sp in ["SOF Δ", "PF Δ", "SOP Δ", "EOP Δ"]:
    pear = df[[sp, "RFS Δ"]].corr(method="pearson").iloc[0, 1]
    spear = df[[sp, "RFS Δ"]].corr(method="spearman").iloc[0, 1]
    print(f"  {sp}: Pearson={pear:+.3f}  Spearman={spear:+.3f}")
    
# Spearman-Korrelationsmatrix der Δ-Spalten auf dem bereinigten 
# Datensatz.
corr_spearman = df[DELTA_SPALTEN].corr(method="spearman")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_spearman, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Spearman-Korrelation der Δ-Spalten")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "03_korrelation_heatmap_SPEARMAN.png"), dpi=150)
plt.show()

# Pearson-Korrelationsmatrix als Vergleichsmaß.
corr_pearson = df[DELTA_SPALTEN].corr(method="pearson")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(corr_pearson, annot=True, fmt=".3f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Pearson-Korrelation der Δ-Spalten")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "04_korrelation_heatmap_PEARSON.png"), dpi=150)
plt.show()

# Wiederholung der Klassifizierungsfunktion aus Kapitel 5.2 zur 
# Berechnung bedingter Wahrscheinlichkeiten auf dem bereinigten 
# Datensatz.
def klassifiziere(x):
    if pd.isna(x):
        return np.nan
    if x < 0:
        return "früh"
    if x == 0:
        return "pünktlich"
    return "verspätet"

for sp in ["SOF Δ", "PF Δ", "SOP Δ", "EOP Δ"]:
    vorgelagert = df[sp].apply(klassifiziere)
    ctab = pd.crosstab(vorgelagert, df["Klassen"], normalize="index") * 100
    print(f"\nP(Klassen | {sp}-Klasse) in %:")
    print(ctab.round(1))


# =============================================================
# 6. Aggregation seltener Ausprägungen kategorialer Features
# 
# Vor dem One-Hot-Encoding werden Ausprägungen, die seltener als 
# MIN_HAEUFIGKEIT (= 10) auftreten, zur Sammelkategorie "other" 
# zusammengefasst. Dies verfolgt zwei Ziele:
# - Vermeidung einer Dimensionalitätsexplosion durch Einzelfälle
# - Vermeidung des Trainings auf Ausprägungen, die im Testdatensatz 
#   mit hoher Wahrscheinlichkeit nicht auftreten und damit keine 
#   generalisierbare Information liefern
# =============================================================

print("\n--- Cardinality vor und nach Aggregation seltener Ausprägungen ---")
for sp in KATEGORIALE_FEATURES:
    haeufigkeiten = df[sp].value_counts()
    seltene = haeufigkeiten[haeufigkeiten < MIN_HAEUFIGKEIT].index
    n_unique_vorher = df[sp].nunique()

    if len(seltene) > 0:
        n_betroffen = df[sp].isin(seltene).sum()
        # Ersetzung der seltenen Ausprägungen durch "other". 
        # Die Konvertierung zu object stellt sicher, dass auch 
        # numerisch typisierte Spalten (z.B. Materialnummer als 
        # float) den String-Wert "other" aufnehmen können.
        df[sp] = df[sp].astype("object").where(~df[sp].isin(seltene), "other")
        n_unique_nachher = df[sp].nunique()
        print(f"  {sp}: {n_unique_vorher} -> {n_unique_nachher} Ausprägungen")
        print(f"    ({len(seltene)} seltene Werte mit insgesamt "
              f"{n_betroffen} Aufträgen zu 'other' zusammengefasst)")
    else:
        print(f"  {sp}: {n_unique_vorher} Ausprägungen "
              f"(keine Aggregation erforderlich)")


# =============================================================
# 7. Diagnose der verworfenen Spalten
# 
# Dokumentation der Spalten, die aufgrund fachlicher Erwägungen oder 
# auf Basis ihrer Kardinalitätsstruktur nicht in den finalen Merk-
# malsvektor übernommen werden. Die Diagnose ist Grundlage für die 
# in Kapitel 5.3 dokumentierte Feature-Selektion und dient gleichzeitig 
# der Nachvollziehbarkeit der Ausschlussentscheidungen.
# =============================================================

print(f"\n--- Diagnose der verworfenen Spalten ---")

# Verworfene kategoriale bzw. string-artige Spalten. Bei 
# Identifikatoren (TPI, Equipment, Sales-Order-Spalten) liegt der 
# überwiegende Teil der Aufträge in seltenen Ausprägungen, sodass 
# keine generalisierbare Information gewonnen werden kann.
verworfene_kategorial = [
    "TPI",
    "Equipment",
    "Business Division",
    "Machine Type",
    "Sales Order - Subsidiary",
    "Sales Order - Central Sales",
    "Sales Order - Production Plant",
    "Ship-to-Party",
]

for sp in verworfene_kategorial:
    if sp not in df.columns:
        print(f"\n  {sp}: nicht im Datensatz vorhanden")
        continue
    n_unique = df[sp].nunique()
    haeufigkeiten = df[sp].value_counts()
    seltene = haeufigkeiten[haeufigkeiten < MIN_HAEUFIGKEIT].index
    n_seltene = len(seltene)
    n_haeufige = n_unique - n_seltene
    n_betroffen_seltene = df[sp].isin(seltene).sum()
    print(f"\n  {sp}")
    print(f"    Unique Werte:           {n_unique}")
    print(f"    Häufige (>= {MIN_HAEUFIGKEIT}):       {n_haeufige}")
    print(f"    Seltene (< {MIN_HAEUFIGKEIT}):        {n_seltene}")
    print(f"    Aufträge in seltenen:   {n_betroffen_seltene} "
          f"({n_betroffen_seltene/len(df)*100:.1f}% des Datensatzes)")

# Verworfene Datums-Spalten: Die einzelnen PLAN- und ACT-Zeitstempel 
# werden nicht direkt übernommen. Stattdessen kommen die daraus 
# abgeleiteten Δ-Werte und Wartezeiten zum Einsatz, die die zeitliche 
# Information in differenzbasierter Form bereitstellen.
print(f"\n  Verworfene Datums-Spalten (PLAN- und ACT-Zeitstempel sowie weitere Datumsfelder):")
verworfene_datum = [
    "Order", "Auftragsanlagedatum", "Anzahlungsrechnungsdatum",
    "SOF PLAN", "SOF ACT", "PF PLAN", "PF ACT",
    "SF PLAN", "SF ACT", "Fertigung Erfassung", "Fertigung Freigabe",
    "SOP PLAN", "SOP ACT", "EOP PLAN", "EOP ACT",
    "RFS PLAN", "RFS ACT",
]
for sp in verworfene_datum:
    if sp in df.columns:
        n_fehlwerte = df[sp].isnull().sum()
        print(f"    {sp:<35}  Fehlwerte: {n_fehlwerte:>4}")


# =============================================================
# 8. Auswahl der finalen Spalten
# 
# Der Merkmalsvektor besteht aus den numerischen Features, den 
# kategorialen Features (vor One-Hot-Encoding) und der Zielvariable. 
# Zusätzlich wird das RFS-ACT-Datum als Hilfsspalte "_split_datum" 
# mitgeführt. Diese Hilfsspalte wird in Kapitel 5.4 ausschließlich 
# für den temporalen 80/20-Split verwendet und vor dem Modelltraining 
# entfernt, sodass kein Data Leakage entsteht.
# =============================================================

df["_split_datum"] = df["RFS ACT"]
spalten_final = (
    NUMERISCHE_FEATURES + KATEGORIALE_FEATURES + [ZIELVARIABLE, "_split_datum"]
)
df = df[spalten_final].copy()
print(f"\nShape nach Spaltenauswahl: {df.shape}")


# =============================================================
# 9. One-Hot-Encoding der kategorialen Features
# 
# Überführung der kategorialen Attribute in binäre Spalten. Für jede 
# Ausprägung wird eine eigene Spalte erzeugt, die den Wert 1 annimmt, 
# wenn die Ausprägung vorliegt, andernfalls 0. Damit wird die 
# qualitative Information für ML-Verfahren nutzbar gemacht, ohne dass 
# eine künstliche Ordnung zwischen den Ausprägungen suggeriert wird.
# =============================================================

n_konzeptionell = len(NUMERISCHE_FEATURES) + len(KATEGORIALE_FEATURES)
print(f"\nKonzeptionelle Features: {n_konzeptionell}")
print(f"  davon numerisch:    {len(NUMERISCHE_FEATURES)}")
print(f"  davon kategorial:   {len(KATEGORIALE_FEATURES)}")

df = pd.get_dummies(df, columns=KATEGORIALE_FEATURES, prefix=KATEGORIALE_FEATURES)

# Technische Spalten-Anzahl im Merkmalsvektor (ohne Zielvariable und Hilfsspalte)
n_technisch = df.shape[1] - 2
print(f"Technische Spalten im Merkmalsvektor "
      f"(ohne Zielvariable und _split_datum): {n_technisch}")
print(f"Shape nach One-Hot-Encoding: {df.shape}")


# =============================================================
# 10. Validierung des bereinigten Datensatzes
# 
# Abschließende Überprüfung der Datentypen und Vollständigkeit. Die 
# methodisch geforderte Vollständigkeit ist erreicht, wenn die Summe 
# der Fehlwerte über alle Spalten null beträgt.
# =============================================================

print("\n--- Datentypen ---")
print(df.dtypes.value_counts())

print("\n--- Fehlwerte gesamt ---")
print(f"Summe Fehlwerte im Datensatz: {df.isnull().sum().sum()}")


# =============================================================
# 11. Export des bereinigten Datensatzes
# 
# Übergabepunkt zwischen Data-Preparation- und Modeling-Phase. Der 
# exportierte Datensatz enthält den vollständigen Merkmalsvektor mit 
# 80 Feature-Spalten, die Zielvariable sowie die Hilfsspalte für den 
# temporalen Split.
# =============================================================

EXPORT_PFAD = os.path.join(OUTPUT_DIR, "02_data_preparation_output.xlsx")
df.to_excel(EXPORT_PFAD, index=False)
print(f"\nDatensatz exportiert: {EXPORT_PFAD}")
print(f"Finale Shape: {df.shape}")

## Modeling

In [ ]:
"""
 Umsetzung der in Kapitel 4.1.3 beschriebenen Methodik:
 - Naive Baseline (Mehrheitsklasse) als Referenzmodell
 - Random Forest pro Prefix-Modell (1 = nach SOF, 2 = nach PF, 
   3 = nach SOP, 4 = nach EOP)
 - Index-Based-Encoding zur Erhaltung der sequenziellen Struktur 
 - Temporaler 80/20-Split nach RFS ACT zur Vermeidung von Data Leakage
 - Zeitseriengerechte Cross-Validation mittels TimeSeriesSplit 
   (rollierender Vorhersageursprung mit expandierendem Trainingsfenster)
 - Grid-Search-Hyperparameter-Tuning mit ROC-AUC im One-vs-Rest-
   Verfahren als Scoring
 - class_weight='balanced' als algorithmische Adressierung des 
   Klassenungleichgewichts
 
 Input:  02_data_preparation_output.xlsx (aus Kapitel 5.3)
 Output: 03_modeling_output/ — finale Modelle, Encoder, Train-/Test-
         set und Trainingskennwerte als Übergabe an die 
         Evaluations-Phase (Kapitel 5.5)
"""


# =============================================================
# Konfiguration
# =============================================================

INPUT_PFAD = "<pfad-zum-output-verzeichnis>/02_data_preparation_output.xlsx"
OUTPUT_DIR = "<pfad-zum-output-verzeichnis>/03_modeling_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ZIELVARIABLE = "Klassen"

# Hilfsspalte mit dem RFS-ACT-Datum, ausschließlich für den temporalen 
# Split verwendet und vor dem Modelltraining aus dem Feature-Vektor 
# entfernt.
SPLIT_DATUM = "_split_datum"


# Anteil der Testmenge entsprechend dem etablierten 80/20-
# Verfahren. Das tatsächliche Verhältnis weicht aufgrund der strikten 
# Datumstrennung leicht von 80/20 ab (vgl. Kapitel 5.4).
TEST_ANTEIL = 0.20

# Reproduzierbarkeit: Fixierter Seed für die Random-Forest-Initialisierung.
RANDOM_STATE = 42

# Anzahl Folds in der TimeSeriesSplit-Cross-Validation.
CV_SPLITS = 5

# Definition der vier Prefix-Modelle gemäß Index-Based-Encoding. Jeder 
# Prefix entspricht einem Vorhersagezeitpunkt entlang der Meilenstein-
# kette und enthält ausschließlich diejenigen Δ-Werte und Wartezeiten, 
# die zu diesem Zeitpunkt prozessual bereits verfügbar sind. Damit 
# wird methodisch ausgeschlossen, dass das Modell Zukunftsinformation 
# nutzt (Data Leakage).
PREFIX_FEATURES = {
    1: {  # nach SOF
        "name": "Prefix 1 (nach SOF)",
        "delta": ["SOF Δ"],
        "wartezeit": ["Wartezeit SOF ACT - Auftrag"],
    },
    2: {  # nach PF
        "name": "Prefix 2 (nach PF)",
        "delta": ["SOF Δ", "PF Δ"],
        "wartezeit": [
            "Wartezeit SOF ACT - Auftrag",
            "Wartezeit PF ACT - SOF ACT",
        ],
    },
    3: {  # nach SOP
        "name": "Prefix 3 (nach SOP)",
        "delta": ["SOF Δ", "PF Δ", "SOP Δ"],
        "wartezeit": [
            "Wartezeit SOF ACT - Auftrag",
            "Wartezeit PF ACT - SOF ACT",
            "Wartezeit SOP ACT - PF ACT",
            "Wartezeit Freigabe - Erfassung",
            "Wartezeit SOP ACT - Freigabe",
        ],
    },
    4: {  # nach EOP
        "name": "Prefix 4 (nach EOP)",
        "delta": ["SOF Δ", "PF Δ", "SOP Δ", "EOP Δ"],
        "wartezeit": [
            "Wartezeit SOF ACT - Auftrag",
            "Wartezeit PF ACT - SOF ACT",
            "Wartezeit SOP ACT - PF ACT",
            "Wartezeit EOP ACT - SOP ACT",
            "Wartezeit Freigabe - Erfassung",
            "Wartezeit SOP ACT - Freigabe",
        ],
    },
}

# Numerische Features, die in jedem Prefix-Modell dauerhaft verfügbar 
# sind. "Delta SOF ACT - Anzahlung" basiert auf dem Anzahlungsdatum, 
# das vor SOF eintritt oder direkt nach SOF und damit zu jedem Vorhersagezeitpunkt vorliegt.
DAUERHAFTE_NUMERISCH = ["Delta SOF ACT - Anzahlung"]

# Hyperparameter-Grid für die Grid-Search. Die Auswahl der vier 
# Hyperparameter folgt der Standardpraxis für Random Forest:
# - n_estimators: Anzahl der Bäume im Ensemble
# - max_depth: maximale Baumtiefe (None = unbegrenzt)
# - min_samples_split: Mindestanzahl Samples für einen Split
# - max_features: Anzahl Features pro Split (Wurzel oder Logarithmus)
# Die 36 resultierenden Kombinationen werden je Prefix-Modell 
# vollständig durchsucht.
PARAM_GRID = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "max_features": ["sqrt", "log2"],
}


# =============================================================
# 1. Datenimport und chronologische Sortierung
# 
# Sortierung nach RFS ACT als Voraussetzung für den temporalen Split 
# und die nachfolgende TimeSeriesSplit-Cross-Validation. Beide Verfahren 
# setzen voraus, dass die Aufträge in chronologischer Reihenfolge 
# vorliegen.
# =============================================================

df = pd.read_excel(INPUT_PFAD)
print(f"Shape Eingangsdatensatz: {df.shape}")

# Chronologische Sortierung nach RFS ACT für temporalen Split
df = df.sort_values(SPLIT_DATUM).reset_index(drop=True)
print(f"Zeitraum: {df[SPLIT_DATUM].min().date()} bis "
      f"{df[SPLIT_DATUM].max().date()}")


# =============================================================
# 2. Temporaler 80/20-Split nach RFS ACT
# 
# Der Split erfolgt strikt nach Datum: Aufträge, die am selben Tag 
# fertiggestellt wurden, werden vollständig einer Seite des Splits 
# zugeordnet. Dadurch wird ausgeschlossen, dass mehrere Aufträge eines 
# Tages zufällig auf beide Mengen verteilt werden, was eine Form von 
# Data Leakage darstellen würde.
# 
# Das resultierende Verhältnis weicht aufgrund dieser strikten 
# Trennung leicht vom 80/20-Ziel ab. Methodisch ist die exakte 
# Datumstrennung jedoch dem präzisen Verhältnis vorzuziehen.
# =============================================================

n_ziel_train = int(len(df) * (1 - TEST_ANTEIL))
split_datum = df.iloc[n_ziel_train][SPLIT_DATUM]
df_train = df[df[SPLIT_DATUM] < split_datum].copy()
df_test = df[df[SPLIT_DATUM] >= split_datum].copy()

print(f"\nSplit-Datum (erstes Datum im Test): {split_datum.date()}")
print(f"Trainingsdatensatz: {len(df_train)} Aufträge "
      f"({df_train[SPLIT_DATUM].min().date()} bis "
      f"{df_train[SPLIT_DATUM].max().date()})")
print(f"Testdatensatz:      {len(df_test)} Aufträge "
      f"({df_test[SPLIT_DATUM].min().date()} bis "
      f"{df_test[SPLIT_DATUM].max().date()})")

# Verifikation: Der Schnittmengenvergleich der Datumswerte muss leer 
# sein. Eine Verletzung dieser Bedingung würde Data Leakage indizieren 
# und wird durch das Assertion abgefangen.
ueberlappung = set(df_train[SPLIT_DATUM]) & set(df_test[SPLIT_DATUM])
assert len(ueberlappung) == 0, (
    f"Data Leakage: {len(ueberlappung)} Datum/Daten in beiden Mengen"
)
print(f"Verifikation: keine Datumsüberlappung zwischen Train und Test.")

# Klassenverteilung in Train- und Testmenge zur Beurteilung der 
# strukturellen Vergleichbarkeit.
print("\nKlassenverteilung Train:")
print((df_train[ZIELVARIABLE].value_counts(normalize=True) * 100).round(2))
print("\nKlassenverteilung Test:")
print((df_test[ZIELVARIABLE].value_counts(normalize=True) * 100).round(2))


# =============================================================
# 3. Label-Encoding der Zielvariable
# 
# Die scikit-learn-Klassifikatoren erwarten numerische Labels. Der 
# LabelEncoder bildet die String-Klassen früh / pünktlich / verspätet 
# auf die Integer-Werte 0 / 1 / 2 ab und wird für die spätere 
# Rücktransformation in die Evaluations-Phase persistiert.
# =============================================================

le = LabelEncoder()
y_train = le.fit_transform(df_train[ZIELVARIABLE])
y_test = le.transform(df_test[ZIELVARIABLE])
print(f"\nLabel-Encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}")



# =============================================================
# 4. Hilfsfunktion zur prefix-spezifischen Feature-Auswahl
# 
# Konkrete Umsetzung des Index-Based-Encodings: Für einen gegebenen 
# Prefix werden nur diejenigen Spalten aus dem vollständigen Merkmals-
# vektor selektiert, die zum entsprechenden Vorhersagezeitpunkt 
# verfügbar sind. Die kategorialen One-Hot-Spalten sind in allen 
# Prefix-Modellen enthalten, da sie statische Auftragsmerkmale 
# repräsentieren.
# =============================================================

def features_fuer_prefix(df_input, prefix_id):
    """Gibt den Feature-Datensatz für einen bestimmten Prefix zurück.
    Enthält nur die zum Vorhersagezeitpunkt verfügbaren Δ-Werte und
    Wartezeiten plus die dauerhaft verfügbaren Features.
    """
    konfig = PREFIX_FEATURES[prefix_id]
    numerisch = konfig["delta"] + konfig["wartezeit"] + DAUERHAFTE_NUMERISCH

    # Identifikation aller One-Hot-Spalten anhand der Präfixe, die in 
    # der Data-Preparation-Phase durch pd.get_dummies erzeugt worden sind.
    onehot_spalten = [
        sp for sp in df_input.columns
        if sp.startswith(("Plant_", "Productfamily_",
                          "Materialnummer_", "Sales Organization_"))
    ]
    spalten = numerisch + onehot_spalten
    return df_input[spalten].copy()


# =============================================================
# 5. Naive Baseline als Referenzmodell
# 
# Der DummyClassifier mit der Strategie "most_frequent" sagt für jeden 
# Auftrag konstant die Mehrheitsklasse des Trainingsdatensatzes 
# vorher. Er definiert die Mindestanforderung, die jedes Random-
# Forest-Modell substanziell übertreffen muss, um einen prädiktiven 
# Mehrwert nachzuweisen. Die Bewertung auf dem Testdatensatz erfolgt 
# in Kapitel 5.5.
# =============================================================

print("\n" + "=" * 60)
print("Naive Baseline (Mehrheitsklasse)")
print("=" * 60)
baseline = DummyClassifier(strategy="most_frequent")

# Der DummyClassifier benötigt formal eine Feature-Matrix, ignoriert 
# deren Inhalt aber. Hier wird die _split_datum-Spalte als technischer 
# Platzhalter übergeben.
X_dummy_train = df_train[[SPLIT_DATUM]]
baseline.fit(X_dummy_train, y_train)
baseline_train_score = baseline.score(X_dummy_train, y_train)

# Identifikation der Mehrheitsklasse zur Dokumentation.
mehrheits_label = baseline.predict(X_dummy_train)[0]
print(f"Mehrheitsklasse: '{le.inverse_transform([mehrheits_label])[0]}' "
      f"(Label {mehrheits_label})")
print(f"Accuracy auf Trainingsdaten: {baseline_train_score:.4f}")

joblib.dump(baseline, os.path.join(OUTPUT_DIR, "baseline.joblib"))


# =============================================================
# 6. Training der Random-Forest-Modelle pro Prefix
# 
# Für jeden der vier Prefixe wird ein separates Random-Forest-Modell 
# trainiert. Das Vorgehen umfasst:
# 1. Auswahl der prefix-spezifischen Features
# 2. Hyperparameter-Tuning mittels GridSearchCV in Verbindung mit 
#    TimeSeriesSplit (zeitseriengerechte 5-Fold-Cross-Validation)
# 3. Refit des finalen Modells mit den besten Hyperparametern auf 
#    dem gesamten Trainingsdatensatz
# 4. Persistierung des Modells für die Evaluations-Phase
# 
# class_weight='balanced' adressiert das Klassenungleichgewicht durch 
# automatische Gewichtung der Klassen invers zu ihrer Häufigkeit.
# =============================================================

ergebnisse = {}

# TimeSeriesSplit realisiert die zeitseriengerechte Cross-Validation 
# nach dem Prinzip des rollierenden Vorhersageursprungs mit 
# expandierendem Trainingsfenster: In jeder Iteration trainiert das 
# Modell auf einer wachsenden Anfangsphase der Trainingsdaten und 
# wird auf der unmittelbar folgenden Periode validiert.
tscv = TimeSeriesSplit(n_splits=CV_SPLITS)

for prefix_id, konfig in PREFIX_FEATURES.items():
    print("\n" + "=" * 60)
    print(f"{konfig['name']}")
    print("=" * 60)

    X_train_prefix = features_fuer_prefix(df_train, prefix_id)
    
    # Aufschlüsselung der Feature-Anzahl zur Dokumentation der 
    # prefix-spezifischen Merkmalsstruktur.
    print(f"Anzahl Features: {X_train_prefix.shape[1]}")
    print(f"  numerisch:     {len(konfig['delta']) + len(konfig['wartezeit']) + len(DAUERHAFTE_NUMERISCH)}")
    print(f"  binär (OHE):   {X_train_prefix.shape[1] - len(konfig['delta']) - len(konfig['wartezeit']) - len(DAUERHAFTE_NUMERISCH)}")

    # Initialisierung des Random Forest mit class_weight='balanced'. 
    # Der RANDOM_STATE sichert die Reproduzierbarkeit der Bootstrap-
    # Stichproben. n_jobs=-1 nutzt alle verfügbaren CPU-Kerne 
    # parallel für das Bauen der Bäume.
    rf = RandomForestClassifier(
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    )
    
    # Grid-Search über das definierte Hyperparameter-Grid. Scoring 
    # ist die ROC-AUC im One-vs-Rest-Verfahren, da die Zielvariable 
    # drei Klassen umfasst und die Arbeit auf eine probabilistische 
    # Vorhersage ausgerichtet ist.
    grid = GridSearchCV(
        estimator=rf,
        param_grid=PARAM_GRID,
        cv=tscv,
        scoring="roc_auc_ovr",
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train_prefix, y_train)

    print(f"\nBeste Hyperparameter: {grid.best_params_}")
    print(f"Beste CV-ROC-AUC:     {grid.best_score_:.4f}")

    # GridSearchCV refittet das Modell mit den besten Hyperparametern 
    # automatisch auf dem gesamten Trainingsdatensatz, sodass 
    # grid.best_estimator_ direkt das finale Modell darstellt.
    finales_modell = grid.best_estimator_

    # Trainings-Accuracy zur Übersicht (Test-Bewertung erfolgt in 5.5)
    train_acc = finales_modell.score(X_train_prefix, y_train)
    print(f"Accuracy Train:       {train_acc:.4f}")

    # Persistierung des finalen Modells für die Evaluations-Phase.
    pfad = os.path.join(OUTPUT_DIR, f"rf_prefix_{prefix_id}.joblib")
    joblib.dump(finales_modell, pfad)
    ergebnisse[prefix_id] = {
        "name": konfig["name"],
        "n_features": int(X_train_prefix.shape[1]),
        "best_params": grid.best_params_,
        "cv_roc_auc_ovr": float(grid.best_score_),
        "train_accuracy": float(train_acc),
    }


# =============================================================
# 7. Zusammenfassung und Persistierung der Ergebnisse
# 
# Tabellarische Übersicht der trainingsbezogenen Kennwerte aller 
# Modelle sowie Persistierung der Datensätze und Encoder für die 
# Evaluations-Phase. Die Trennung von Modellauswahl (5.4) und 
# Modellbewertung (5.5) entspricht der methodischen Anforderung, 
# Generalisierungsleistung ausschließlich auf einem Datensatz zu 
# bewerten, der weder zum Training noch zum Tuning herangezogen worden ist.
# =============================================================

print("\n" + "=" * 60)
print("Zusammenfassung Modeling-Phase")
print("=" * 60)
print(f"{'Modell':<25} {'Features':>10} {'CV AUC':>10} {'Train Acc':>12}")
print(f"{'Baseline':<25} {'-':>10} {'-':>10} "
      f"{baseline_train_score:>12.4f}")
for prefix_id, res in ergebnisse.items():
    print(f"{res['name']:<25} {res['n_features']:>10} "
          f"{res['cv_roc_auc_ovr']:>10.4f} {res['train_accuracy']:>12.4f}")

# Trainings- und Testset für die Evaluation in 5.5 speichern
df_train.to_pickle(os.path.join(OUTPUT_DIR, "df_train.pkl"))
df_test.to_pickle(os.path.join(OUTPUT_DIR, "df_test.pkl"))
joblib.dump(le, os.path.join(OUTPUT_DIR, "label_encoder.joblib"))

# JSON-Export der trainingsbezogenen Kennwerte als menschenlesbare 
# Dokumentation. Die Zusammenfassung enthält ausschließlich Werte, 
# die in der Modeling-Phase ermittelt worden sind. Test-Metriken werden 
# erst in Kapitel 5.5 hinzugefügt.
ergebnisse_export = {
    "baseline": {
        "train_accuracy": float(baseline_train_score),
    },
    "label_encoding": {cls: int(i) for cls, i in zip(le.classes_, le.transform(le.classes_))},
    "split": {
        "n_train": len(df_train),
        "n_test": len(df_test),
        "split_datum": str(split_datum.date()),
        "train_zeitraum": [
            str(df_train[SPLIT_DATUM].min().date()),
            str(df_train[SPLIT_DATUM].max().date()),
        ],
        "test_zeitraum": [
            str(df_test[SPLIT_DATUM].min().date()),
            str(df_test[SPLIT_DATUM].max().date()),
        ],
    },
    "prefix_modelle": ergebnisse,
}
with open(os.path.join(OUTPUT_DIR, "modeling_zusammenfassung.json"),
          "w", encoding="utf-8") as f:
    json.dump(ergebnisse_export, f, indent=2, ensure_ascii=False)

print(f"\nModelle, Train-/Testset und Zusammenfassung gespeichert in: {OUTPUT_DIR}")

## Demonstration und Evaluation 

In [ ]:
"""
 Umsetzung der in Kapitel 4.2 beschriebenen Methodik:
 - Demonstration durch vollständigen Pipeline-Durchlauf auf dem 
   unabhängigen Testdatensatz, der die reale Anwendungssituation 
   im Produktionsbetrieb wirklichkeitsgetreu widerspiegelt
 - Klassifikationsmetriken (Accuracy, Precision, Recall, F1)
 - ROC-AUC im One-vs-Rest-Verfahren als schwellenwertunabhängige, 
   probabilistische Metrik
 - Vergleich der vier Prefix-Modelle gegen die naive Baseline 
   zum Nachweis des informationellen Mehrwerts
 - Bewertung über alle vier Vorhersagezeitpunkte zur Beurteilung 
   der Earliness-Eigenschaft
 - SHAP-basierte Modellinterpretation auf globaler und lokaler 
   Ebene zur Erfüllung der Erklärbarkeitsanforderung
 
 Input:  03_modeling_output/ (aus Kapitel 5.4)
 Output: 04_evaluation_output/ — Metriken, Konfusionsmatrizen, 
         SHAP-Plots und JSON-Zusammenfassung
"""


# =============================================================
# Konfiguration
# =============================================================

INPUT_DIR = "<pfad-zum-output-verzeichnis>/03_modeling_output"
OUTPUT_DIR = "<pfad-zum-output-verzeichnis>/04_evaluation_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

ZIELVARIABLE = "Klassen"
SPLIT_DATUM = "_split_datum"

# Identische Prefix-Definitionen wie in Kapitel 5.4. Die Wiederholung 
# stellt sicher, dass die Feature-Selektion in der Evaluations-Phase 
# konsistent zur Trainings-Phase erfolgt.
PREFIX_FEATURES = {
    1: {
        "name": "Prefix 1 (nach SOF)",
        "delta": ["SOF Δ"],
        "wartezeit": ["Wartezeit SOF ACT - Auftrag"],
    },
    2: {
        "name": "Prefix 2 (nach PF)",
        "delta": ["SOF Δ", "PF Δ"],
        "wartezeit": [
            "Wartezeit SOF ACT - Auftrag",
            "Wartezeit PF ACT - SOF ACT",
        ],
    },
    3: {
        "name": "Prefix 3 (nach SOP)",
        "delta": ["SOF Δ", "PF Δ", "SOP Δ"],
        "wartezeit": [
            "Wartezeit SOF ACT - Auftrag",
            "Wartezeit PF ACT - SOF ACT",
            "Wartezeit SOP ACT - PF ACT",
            "Wartezeit Freigabe - Erfassung",
            "Wartezeit SOP ACT - Freigabe",
        ],
    },
    4: {
        "name": "Prefix 4 (nach EOP)",
        "delta": ["SOF Δ", "PF Δ", "SOP Δ", "EOP Δ"],
        "wartezeit": [
            "Wartezeit SOF ACT - Auftrag",
            "Wartezeit PF ACT - SOF ACT",
            "Wartezeit SOP ACT - PF ACT",
            "Wartezeit EOP ACT - SOP ACT",
            "Wartezeit Freigabe - Erfassung",
            "Wartezeit SOP ACT - Freigabe",
        ],
    },
}
DAUERHAFTE_NUMERISCH = ["Delta SOF ACT - Anzahlung"]


# =============================================================
# 1. Persistierte Modelle und Datensätze laden
# 
# Die in Kapitel 5.4 trainierten Modelle, der LabelEncoder sowie 
# die Train- und Testsets werden geladen. Damit ist gewährleistet, 
# dass die Evaluation auf exakt denselben Daten und Modellen erfolgt, 
# auf denen auch das Tuning durchgeführt worden ist.
# =============================================================

print("=" * 60)
print("Demonstration und Evaluation - Kapitel 5.5")
print("=" * 60)

df_train = pd.read_pickle(os.path.join(INPUT_DIR, "df_train.pkl"))
df_test = pd.read_pickle(os.path.join(INPUT_DIR, "df_test.pkl"))
le = joblib.load(os.path.join(INPUT_DIR, "label_encoder.joblib"))
baseline = joblib.load(os.path.join(INPUT_DIR, "baseline.joblib"))

print(f"\nTrainingsdatensatz: {len(df_train)} Aufträge")
print(f"Testdatensatz:      {len(df_test)} Aufträge")
print(f"Klassen:            {list(le.classes_)}")

y_train = le.transform(df_train[ZIELVARIABLE])
y_test = le.transform(df_test[ZIELVARIABLE])


# =============================================================
# 2. Hilfsfunktion zur prefix-spezifischen Feature-Auswahl
# 
# Identische Logik wie in der Modeling-Phase. Die Wiederverwendung 
# stellt sicher, dass für die Evaluation exakt dieselben 
# Feature-Vektoren konstruiert werden wie für das Training.
# =============================================================

def features_fuer_prefix(df_input, prefix_id):
    konfig = PREFIX_FEATURES[prefix_id]
    numerisch = konfig["delta"] + konfig["wartezeit"] + DAUERHAFTE_NUMERISCH
    onehot_spalten = [
        sp for sp in df_input.columns
        if sp.startswith(("Plant_", "Productfamily_",
                          "Materialnummer_", "Sales Organization_"))
    ]
    return df_input[numerisch + onehot_spalten].copy()


# =============================================================
# 3. Hilfsfunktion zur Berechnung der Klassifikationsmetriken
# 
# Berechnet sowohl Macro-gemittelte als auch klassenspezifische 
# Werte für Precision, Recall und F1-Score. Die Macro-Mittelung 
# gewichtet jede Klasse gleich, unabhängig von ihrer Häufigkeit, 
# was bei leicht unausgewogenen Klassenverteilungen eine fairere 
# Gesamtbewertung liefert als die mikrogemittelte Variante.
# 
# Die ROC-AUC wird nur berechnet, wenn Wahrscheinlichkeitsschätzungen 
# vorliegen. Da die naive Baseline keine differenzierten 
# Wahrscheinlichkeiten liefert, wird sie ohne ROC-AUC bewertet.
# =============================================================

def metriken_berechnen(y_true, y_pred, y_proba=None):
    """Berechnet Standard-Klassifikationsmetriken auf Macro-Ebene
    plus klassenspezifisch. ROC AUC nur, falls Wahrscheinlichkeiten vorliegen."""
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    # Berechnung der Metriken zusätzlich auf Klassenebene zur 
    # differenzierten Bewertung der Modellgüte je Zielklasse.
    p_klassen = precision_score(y_true, y_pred, average=None, zero_division=0)
    r_klassen = recall_score(y_true, y_pred, average=None, zero_division=0)
    f_klassen = f1_score(y_true, y_pred, average=None, zero_division=0)
    metrics["klassenspezifisch"] = {
        cls: {
            "precision": float(p_klassen[i]),
            "recall": float(r_klassen[i]),
            "f1": float(f_klassen[i]),
        }
        for i, cls in enumerate(le.classes_)
    }
    if y_proba is not None:
        metrics["roc_auc_ovr"] = float(
            roc_auc_score(y_true, y_proba, multi_class="ovr", average="macro")
        )
    return metrics


# =============================================================
# 4. Bewertung der naiven Baseline auf dem Testdatensatz
# 
# Die Baseline definiert die Mindestanforderung, die jedes Prefix-
# Modell substanziell übertreffen muss. Da sie für jeden Auftrag 
# konstant die Mehrheitsklasse "pünktlich" vorhersagt, fallen die 
# klassenspezifischen Metriken für "früh" und "verspätet" 
# definitionsgemäß auf null.
# =============================================================

print("\n" + "=" * 60)
print("Naive Baseline auf Testdatensatz")
print("=" * 60)
X_dummy_test = df_test[[SPLIT_DATUM]]
y_pred_baseline = baseline.predict(X_dummy_test)
baseline_metrics = metriken_berechnen(y_test, y_pred_baseline)
print(f"Accuracy:        {baseline_metrics['accuracy']:.4f}")
print(f"Precision Macro: {baseline_metrics['precision_macro']:.4f}")
print(f"Recall Macro:    {baseline_metrics['recall_macro']:.4f}")
print(f"F1 Macro:        {baseline_metrics['f1_macro']:.4f}")


# =============================================================
# 5. Bewertung der Prefix-Modelle auf dem Testdatensatz
# 
# Für jedes der vier Prefix-Modelle werden die Vorhersagen auf dem 
# Testdatensatz erzeugt und die Metriken berechnet. 
# Der classification_report ergänzt die numerischen Metriken um eine 
# kompakte Übersicht je Klasse.
# =============================================================

ergebnisse = {"baseline": baseline_metrics}

for prefix_id, konfig in PREFIX_FEATURES.items():
    print("\n" + "=" * 60)
    print(f"{konfig['name']} auf Testdatensatz")
    print("=" * 60)

    modell = joblib.load(os.path.join(INPUT_DIR, f"rf_prefix_{prefix_id}.joblib"))
    X_test_prefix = features_fuer_prefix(df_test, prefix_id)
    y_pred = modell.predict(X_test_prefix)
    y_proba = modell.predict_proba(X_test_prefix)

    metrics = metriken_berechnen(y_test, y_pred, y_proba)
    print(f"Accuracy:        {metrics['accuracy']:.4f}")
    print(f"Precision Macro: {metrics['precision_macro']:.4f}")
    print(f"Recall Macro:    {metrics['recall_macro']:.4f}")
    print(f"F1 Macro:        {metrics['f1_macro']:.4f}")
    print(f"ROC AUC OvR:     {metrics['roc_auc_ovr']:.4f}")

    print("\nKlassenspezifisch:")
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

    ergebnisse[f"prefix_{prefix_id}"] = metrics


# =============================================================
# 6. Konfusionsmatrizen
# 
# Visualisierung der Verteilung von Vorhersagen und tatsächlichen 
# Klassen für alle vier Prefix-Modelle. Konfusionsmatrizen erlauben 
# eine differenzierte Beurteilung, welche Klassen miteinander 
# verwechselt werden, was über die aggregierten Metriken hinausgeht.
# =============================================================

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for idx, (prefix_id, konfig) in enumerate(PREFIX_FEATURES.items()):
    modell = joblib.load(os.path.join(INPUT_DIR, f"rf_prefix_{prefix_id}.joblib"))
    X_test_prefix = features_fuer_prefix(df_test, prefix_id)
    y_pred = modell.predict(X_test_prefix)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=le.classes_, yticklabels=le.classes_,
        ax=axes[idx], cbar=False
    )
    axes[idx].set_title(konfig["name"])
    axes[idx].set_xlabel("Vorhergesagt")
    axes[idx].set_ylabel("Tatsächlich")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "konfusionsmatrizen.png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"\nKonfusionsmatrizen gespeichert: {OUTPUT_DIR}\\konfusionsmatrizen.png")



# =============================================================
# 7. Earliness-Vergleich
# 
# Visualisierung der Klassifikationsleistung über die vier 
# Vorhersagezeitpunkte. --> Visualisierung wird für Arbeit nicht
# genutzt, sondern erklärt (nur zur Überprüfung erstellt)
# =============================================================

prefix_namen = ["Baseline"] + [PREFIX_FEATURES[i]["name"] for i in range(1, 5)]
prefix_acc = [baseline_metrics["accuracy"]] + [
    ergebnisse[f"prefix_{i}"]["accuracy"] for i in range(1, 5)
]
prefix_f1 = [baseline_metrics["f1_macro"]] + [
    ergebnisse[f"prefix_{i}"]["f1_macro"] for i in range(1, 5)
]
prefix_auc = [None] + [
    ergebnisse[f"prefix_{i}"]["roc_auc_ovr"] for i in range(1, 5)
]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(prefix_namen))
breite = 0.25

ax.bar(x - breite, prefix_acc, breite, label="Accuracy", color="steelblue")
ax.bar(x, prefix_f1, breite, label="F1 Macro", color="darkorange")

# ROC-AUC wird nur für die Prefix-Modelle dargestellt, da die Baseline 
# keine differenzierten Wahrscheinlichkeitsschätzungen liefert.
auc_werte = [v for v in prefix_auc if v is not None]
ax.bar(x[1:] + breite, auc_werte, breite, label="ROC AUC OvR", color="seagreen")

ax.set_xticks(x)
ax.set_xticklabels(prefix_namen, rotation=15, ha="right")
ax.set_ylabel("Metrikwert")
ax.set_ylim(0, 1)
ax.set_title("Earliness-Vergleich der Prefix-Modelle")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "earliness_vergleich.png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"Earliness-Vergleich gespeichert: {OUTPUT_DIR}\\earliness_vergleich.png")


# =============================================================
# 8. Globale SHAP-Auswertung pro Prefix-Modell
# 
# Der globale Summary Plot zeigt den mittleren absoluten SHAP-Wert 
# je Feature über alle Aufträge des Testdatensatzes hinweg, 
# aufgeschlüsselt nach den drei Zielklassen. Damit wird sichtbar, 
# welche Features in welcher Klasse besonders einflussreich sind.
# =============================================================

print("\n" + "=" * 60)
print("SHAP-Auswertung")
print("=" * 60)

# Speicherung der SHAP-Werte für die spätere lokale Auswertung. 
shap_werte_speicher = {}

for prefix_id, konfig in PREFIX_FEATURES.items():
    print(f"\n{konfig['name']}: SHAP berechnen ...")
    modell = joblib.load(os.path.join(INPUT_DIR, f"rf_prefix_{prefix_id}.joblib"))
    X_test_prefix = features_fuer_prefix(df_test, prefix_id)

    # TreeExplainer für Random Forest
    explainer = shap.TreeExplainer(modell)
    shap_werte = explainer.shap_values(X_test_prefix)

    # shap_values bei Multi-Class Random Forest
    if isinstance(shap_werte, list):
       
        shap_werte_3d = np.stack(shap_werte, axis=-1)
    else:
        shap_werte_3d = shap_werte

    shap_werte_speicher[prefix_id] = {
        "shap_3d": shap_werte_3d,
        "X_test": X_test_prefix,
        "explainer": explainer,
    }

    # Globaler Summary Plot mit den 15 wichtigsten Features. Die 
    # explizite xlabel-Setzung überschreibt die standardmäßig 
    # langgezogene shap-Beschriftung und verbessert die Lesbarkeit.
    plt.figure(figsize=(12, 8))
    shap.summary_plot(
        shap_werte_3d, X_test_prefix,
        plot_type="bar", class_names=list(le.classes_),
        max_display=15, show=False
    )
    plt.title(f"SHAP Global Importance - {konfig['name']}")
    plt.xlabel("mean(|SHAP value|)")
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, f"shap_global_prefix_{prefix_id}.png"),
        dpi=150, bbox_inches="tight"
    )
    plt.close()
    print(f"  Global gespeichert: shap_global_prefix_{prefix_id}.png")


# =============================================================
# 9. Auswahl der lokalen SHAP-Beispiele
# 
# Für jede der drei Zielklassen wird derjenige Auftrag aus dem 
# Testdatensatz ausgewählt, den das Prefix-4-Modell mit der höchsten 
# Vorhersagewahrscheinlichkeit korrekt klassifiziert hat. Die 
# Auswahl auf Basis von Prefix 4 erfolgt, weil dieses Modell den 
# höchsten Informationsgehalt aufweist und damit die saubersten 
# Beispiele liefert.
# 
# Methodische Anmerkung: Die gezielte Auswahl korrekt klassifizierter 
# Aufträge mit hoher Vorhersagewahrscheinlichkeit dient der 
# Demonstration der Modelllogik, nicht einer repräsentativen 
# Fehleranalyse.
# =============================================================

print("\nLokale SHAP-Beispiele (höchste Vorhersagesicherheit pro Klasse):")

# Auswahl der Beispiele basiert auf Prefix 4 (bestes Modell)
modell_p4 = joblib.load(os.path.join(INPUT_DIR, "rf_prefix_4.joblib"))
X_test_p4 = features_fuer_prefix(df_test, 4)
y_proba_p4 = modell_p4.predict_proba(X_test_p4)
y_pred_p4 = modell_p4.predict(X_test_p4)

beispiele = {}
for cls_idx, cls_name in enumerate(le.classes_):
    
    # Nur korrekt klassifizierte Aufträge der jeweiligen Klasse
    korrekt_mask = (y_test == cls_idx) & (y_pred_p4 == cls_idx)
    if korrekt_mask.sum() == 0:
        print(f"  '{cls_name}': keine korrekt klassifizierten Aufträge gefunden")
        continue
    
    # Innerhalb dieser Aufträge denjenigen mit höchster Wahrscheinlichkeit für die Klasse
    proba_klasse = y_proba_p4[:, cls_idx]
    proba_klasse_korrekt = np.where(korrekt_mask, proba_klasse, -np.inf)
    auswahl_idx = int(np.argmax(proba_klasse_korrekt))

    beispiele[cls_name] = {
        "test_index": auswahl_idx,
        "wahrscheinlichkeit": float(proba_klasse[auswahl_idx]),
    }
    print(f"  '{cls_name}': Test-Index {auswahl_idx}, "
          f"P({cls_name}) = {proba_klasse[auswahl_idx]:.4f}")


# =============================================================
# 10. Auftrags-Diagnose der lokalen Beispiele
# 
# Für jedes der drei ausgewählten Beispiele werden die Wahrscheinlich-
# keitsschätzungen aller drei Klassen über alle vier Prefix-Modelle 
# dokumentiert. Damit wird transparent, wie sich die Vorhersage des 
# einzelnen Auftrags entlang der Meilensteinkette entwickelt und 
# inwiefern die Klassifikationsentscheidung bereits in frühen 
# Prefix-Modellen stabil ist.
# =============================================================

print("\n" + "=" * 60)
print("Auftrags-Diagnose der lokalen SHAP-Beispiele")
print("=" * 60)

diagnose = {}
for cls_name, info in beispiele.items():
    test_idx = info["test_index"]
    tatsaechliche_klasse = le.inverse_transform([y_test[test_idx]])[0]

    print(f"\nBeispiel: tatsächliche Klasse '{tatsaechliche_klasse}' "
          f"(Test-Index {test_idx})")
    print(f"{'Prefix':<22} {'P(früh)':>10} {'P(pünktlich)':>14} "
          f"{'P(verspätet)':>14} {'Vorhersage':>14} {'Treffer':>10}")

    diagnose[cls_name] = {
        "test_index": test_idx,
        "tatsaechliche_klasse": tatsaechliche_klasse,
        "prefix_modelle": {},
    }
    
    # Für jedes Prefix-Modell die Wahrscheinlichkeitsverteilung 
    # ermitteln und gegen die tatsächliche Klasse abgleichen.
    for prefix_id, konfig in PREFIX_FEATURES.items():
        modell = joblib.load(os.path.join(INPUT_DIR, f"rf_prefix_{prefix_id}.joblib"))
        X_test_prefix = features_fuer_prefix(df_test, prefix_id)
        proba = modell.predict_proba(X_test_prefix)[test_idx]
        vorhersage_idx = int(np.argmax(proba))
        vorhersage_klasse = le.inverse_transform([vorhersage_idx])[0]
        treffer = "ja" if vorhersage_klasse == tatsaechliche_klasse else "nein"

        print(f"{konfig['name']:<22} "
              f"{proba[0]:>10.4f} {proba[1]:>14.4f} {proba[2]:>14.4f} "
              f"{vorhersage_klasse:>14} {treffer:>10}")

        diagnose[cls_name]["prefix_modelle"][prefix_id] = {
            "P_früh": float(proba[0]),
            "P_pünktlich": float(proba[1]),
            "P_verspätet": float(proba[2]),
            "vorhersage": vorhersage_klasse,
            "treffer": treffer == "ja",
        }


# =============================================================
# 11. Lokale SHAP-Plots
# 
# Visualisierung der Top-10-Features mit dem stärksten Einfluss auf 
# die Vorhersagewahrscheinlichkeit der jeweiligen Zielklasse, getrennt 
# nach den vier Prefix-Modellen. Positive SHAP-Werte erhöhen die 
# Klassen-Wahrscheinlichkeit, negative mindern sie. Die farbliche 
# Codierung (grün = positiver Beitrag, rot = negativer Beitrag) 
# unterstützt die intuitive Interpretation.
# =============================================================

for cls_name, info in beispiele.items():
    fig, axes = plt.subplots(1, 4, figsize=(28, 8))
    test_idx = info["test_index"]
    cls_idx = list(le.classes_).index(cls_name)

    for ax_idx, prefix_id in enumerate([1, 2, 3, 4]):
        speicher = shap_werte_speicher[prefix_id]
        shap_3d = speicher["shap_3d"]
        X_prefix = speicher["X_test"]

        # SHAP-Werte für die Zielklasse extrahieren
        shap_values_klasse = shap_3d[test_idx, :, cls_idx]
        feature_werte = X_prefix.iloc[test_idx].values

        # Top-10-Features nach absolutem SHAP-Wert
        importance = np.abs(shap_values_klasse)
        top10_idx = np.argsort(importance)[-10:]
        top10_werte = shap_values_klasse[top10_idx]
        top10_namen = X_prefix.columns[top10_idx]
        top10_feature_werte = feature_werte[top10_idx]
        
        # Farbcodierung der Balken nach Wirkungsrichtung
        farben = ["seagreen" if v > 0 else "indianred" for v in top10_werte]
        ax = axes[ax_idx]
        ax.barh(range(len(top10_werte)), top10_werte, color=farben)
        ax.set_yticks(range(len(top10_werte)))
        
        # Labels enthalten Featurename und tatsächlichen Wert für 
        # den jeweiligen Auftrag, was die Interpretation erleichtert.
        ax.set_yticklabels(
            [f"{n} = {w}" for n, w in zip(top10_namen, top10_feature_werte)],
            fontsize=8
        )
        ax.axvline(0, color="black", linewidth=0.5)
        ax.set_xlabel(f"SHAP-Wert für Klasse '{cls_name}'")
        ax.set_title(PREFIX_FEATURES[prefix_id]["name"])
        ax.grid(axis="x", alpha=0.3)

    plt.suptitle(f"Lokale SHAP-Erklärung: tatsächliche Klasse '{cls_name}' "
                 f"(Test-Index {test_idx})", fontsize=12)
    plt.tight_layout()
    pfad = os.path.join(OUTPUT_DIR, f"shap_lokal_{cls_name}.png")
    plt.savefig(pfad, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Lokal gespeichert: shap_lokal_{cls_name}.png")


# =============================================================
# 12. Zusammenfassende Übersichtstabelle
# 
# Tabellarische Aggregation aller Bewertungsmetriken über die fünf 
# evaluierten Modelle (Baseline plus vier Prefix-Modelle)
# =============================================================

print("\n" + "=" * 60)
print("Zusammenfassung Demonstration und Evaluation")
print("=" * 60)
print(f"{'Modell':<25} {'Acc':>8} {'Prec':>8} {'Rec':>8} {'F1':>8} {'AUC':>8}")
print(f"{'Baseline':<25} "
      f"{baseline_metrics['accuracy']:>8.4f} "
      f"{baseline_metrics['precision_macro']:>8.4f} "
      f"{baseline_metrics['recall_macro']:>8.4f} "
      f"{baseline_metrics['f1_macro']:>8.4f} "
      f"{'-':>8}")
for prefix_id, konfig in PREFIX_FEATURES.items():
    m = ergebnisse[f"prefix_{prefix_id}"]
    print(f"{konfig['name']:<25} "
          f"{m['accuracy']:>8.4f} "
          f"{m['precision_macro']:>8.4f} "
          f"{m['recall_macro']:>8.4f} "
          f"{m['f1_macro']:>8.4f} "
          f"{m['roc_auc_ovr']:>8.4f}")


# =============================================================
# 13. JSON-Export aller Evaluationsergebnisse
# 
# Strukturierte Persistierung sämtlicher Metriken, lokaler 
# Beispielindizes und Auftragsdiagnosen als menschenlesbare 
# Dokumentation der Evaluationsphase.
# =============================================================

ergebnisse_export = {
    "baseline": baseline_metrics,
    "prefix_modelle": {
        f"prefix_{i}": ergebnisse[f"prefix_{i}"] for i in range(1, 5)
    },
    "lokale_beispiele": {cls: info for cls, info in beispiele.items()},
    "auftrags_diagnose": diagnose,
}
with open(os.path.join(OUTPUT_DIR, "evaluation_zusammenfassung.json"),
          "w", encoding="utf-8") as f:
    json.dump(ergebnisse_export, f, indent=2, ensure_ascii=False)

print(f"\nAlle Ergebnisse gespeichert in: {OUTPUT_DIR}")